In [1]:
from collections import Counter
from pathlib import Path
import pickle
import automated_llm_probes as alp

TARGET_N = 600
NAMES = ['claude-haiku-4.5',
 'claude-opus-4.5',
 'claude-opus-4.7',
 'claude-opus-5',
 'claude-sonnet-4.5',
 'gpt-3.5-turbo',
 'gpt-4-turbo',
 'gpt-4-turbo',
 'gpt-4o',
 'gpt-4o-mini',
 'gpt-5.4',
 'grok-4.2',
 'grok-4.3',
 'grok-4.5',
 'grok-build-0.1',
 'llama-3.1-8b',
 'llama-3.2-3b',
 'llama-4-guard-12b',
 'llama-4-maverick',
 'llama-4-scout']

HUMAN_N = {"brick": 2019, "knife": 1028, "car tires": 960, "box": 833, "rope": 829,
    "pen": 742, "wooden slat": 671, "paperclip": 534, "tin can": 425,
    "socks": 339, "light bulb": 337, "spoon": 337, "towel": 327, "book": 326,
    "belt": 300, "bucket": 300, "sock": 300, "candle": 299}

def targets(n):
    tot = sum(HUMAN_N.values())
    raw = {c: n * k / tot for c, k in HUMAN_N.items()}
    out = {c: int(v) for c, v in raw.items()}
    for c in sorted(raw, key=lambda c: raw[c] - out[c], reverse=True):
        if sum(out.values()) >= n:
            break
        out[c] += 1
    return out

tgt = targets(TARGET_N)
models = [m for m in alp.ready_models() if m["name"] in NAMES]
print("N", TARGET_N, [m["name"] for m in models])

for m in models:
    have = Counter()
    for p in Path("aut", m["name"]).rglob("*.pickle"):
        try:
            row = pickle.load(open(p, "rb"))
        except Exception:
            continue
        cue = (row.get("kwargs") or {}).get("cue") or row.get("cue")
        if cue:
            have[str(cue).strip().lower()] += 1
    n_have = sum(have.values())
    print(f"\n{m['name']}  {n_have}")
    for cue, want in tgt.items():
        need = max(0, want - have.get(cue, 0))
        print(f"  {cue:16s} {have.get(cue, 0):4d}/{want:<3d}  {'ok' if need == 0 else f'+{need}'}")
        if need:
            alp.collect("AUT", models=[m], n_per_model=n_have + need, cue=cue)
            n_have += need

N 600 ['grok-4.2', 'grok-4.3', 'grok-4.5', 'grok-build-0.1', 'gpt-3.5-turbo', 'gpt-4-turbo', 'gpt-4o', 'gpt-5.4', 'gpt-4o-mini', 'gpt-4-turbo', 'llama-4-guard-12b', 'llama-4-scout', 'llama-4-maverick', 'llama-3.2-3b', 'llama-3.1-8b', 'claude-sonnet-4.5', 'claude-haiku-4.5', 'claude-opus-4.5', 'claude-opus-4.7', 'claude-opus-5']

grok-4.2  0
  brick               0/111  +111
  grok-4.2: 620/111 done — skip
  knife               0/57   +57
  grok-4.2: 620/168 done — skip
  car tires           0/53   +53
  grok-4.2: 620/221 done — skip
  box                 0/46   +46
  grok-4.2: 620/267 done — skip
  rope                0/46   +46
  grok-4.2: 620/313 done — skip
  pen                 0/41   +41
  grok-4.2: 620/354 done — skip
  wooden slat         0/37   +37
  grok-4.2: 620/391 done — skip
  paperclip           0/29   +29
  grok-4.2: 620/420 done — skip
  tin can             0/23   +23
  grok-4.2: 620/443 done — skip
  socks               0/19   +19
  grok-4.2: 620/462 done — skip
  ligh

  gpt-4o: 681/462 done — skip
  light bulb          0/19   +19
  gpt-4o: 681/481 done — skip
  spoon               0/19   +19
  gpt-4o: 681/500 done — skip
  towel               0/18   +18
  gpt-4o: 681/518 done — skip
  book                0/18   +18
  gpt-4o: 681/536 done — skip
  belt                0/16   +16
  gpt-4o: 681/552 done — skip
  bucket              0/16   +16
  gpt-4o: 681/568 done — skip
  sock                0/16   +16
  gpt-4o: 681/584 done — skip
  candle              0/16   +16
  gpt-4o: 681/600 done — skip

gpt-5.4  0
  brick               0/111  +111
  gpt-5.4: 618/111 done — skip
  knife               0/57   +57
  gpt-5.4: 618/168 done — skip
  car tires           0/53   +53
  gpt-5.4: 618/221 done — skip
  box                 0/46   +46
  gpt-5.4: 618/267 done — skip
  rope                0/46   +46
  gpt-5.4: 618/313 done — skip
  pen                 0/41   +41
  gpt-5.4: 618/354 done — skip
  wooden slat         0/37   +37
  gpt-5.4: 618/391 done — skip
  pap

AUT: 100%|█████████████████████████████████████████████████████████████████| 4/4 [00:03<00:00,  1.22it/s]


  pen                 0/41   +41
  llama-4-guard-12b: 313 collected, 41 to collect


AUT: 100%|███████████████████████████████████████████████████████████████| 41/41 [00:32<00:00,  1.28it/s]


...

AUT: 100%|███████████████████████████████████████████████████████████████| 16/16 [04:17<00:00, 16.11s/it]


  sock                0/16   +16
  claude-opus-5: 568 collected, 16 to collect


AUT: 100%|███████████████████████████████████████████████████████████████| 16/16 [05:27<00:00, 20.49s/it]


  candle              0/16   +16
  claude-opus-5: 584 collected, 16 to collect


AUT: 100%|███████████████████████████████████████████████████████████████| 16/16 [05:01<00:00, 18.84s/it]
